<h2> Setup </h2>

In [33]:
from data.data import Data
from clients.client import Client
from models.LeNet import LeNet
from algorithms.per_fedAvg import PerFedAvg
from algorithms.fedAvg import FedAvg
from attacks.iDLG import iDLG
import numpy as np
import tensorflow as tf

In [34]:
ds = Data()
client1 = Client(1, ds, batch_size=1)
model = LeNet()
attack = iDLG()

W0000 00:00:1776180109.004798 47744343 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


<h2> Measurement </h2>

In [35]:
def mse_measurement(used_in_training_data, attack):
    results = []
    for (x, y) in used_in_training_data:
        actual_input = np.array(x)
        reconstructed_input = np.array(attack.reconstructed_input)
        input_mse = float(np.mean((reconstructed_input - actual_input) ** 2))

        actual_label = int(y.numpy()[0]) if hasattr(y, 'numpy') else int(y[0])
        label_mse = float((int(attack.reconstructed_label) - actual_label) ** 2)

        results.append({"input_mse": input_mse, "label_mse": label_mse})
    return results

<h2> Run Simulation </h2>

In [36]:
per_fedAvg = PerFedAvg(model.clone(), [client1], {
    "communication_rounds": 1,
    "client_training_rounds": 1,
    "client_adaptation_rounds": 1,
    "client_training_batch_size": 1,
    "reuse_data_batches": True, 
    "loss_function": tf.keras.losses.SparseCategoricalCrossentropy(),
})

per_fedavg_results = per_fedAvg.run(attack=iDLG(), performance_metrics=[mse_measurement])

W0000 00:00:1776180120.324496 47744882 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Communication round 1 of 1
Clients model training completed
Clients update aggregation completed


<h2> Results </h2>

In [37]:
print("=== Per-FedAvg + iDLG ===")
for i, metric_result in enumerate(per_fedavg_results):
    print(f"Metric result {i + 1}:")
    for sample_result in metric_result:
        print(f"  Input MSE:  {sample_result['input_mse']}")
        print(f"  Label MSE:  {sample_result['label_mse']}")

=== Per-FedAvg + iDLG ===
Metric result 1:
  Input MSE:  11544.4091796875
  Label MSE:  16.0


<h2> FedAvg + iDLG </h2>

In [38]:
fedavg = FedAvg(model.clone(), [Client(2, ds, batch_size=1)], {
    "communication_rounds": 1,
    "client_training_rounds": 1,
    "client_training_batch_size": 1,
    "loss_function": tf.keras.losses.SparseCategoricalCrossentropy(),
})

fedavg_results = fedavg.run(attack=iDLG(), performance_metrics=[mse_measurement])

I0000 00:00:1776180504.114651 47766246 shuffle_dataset_op.cc:453] ShuffleDatasetV3:252: Filling up shuffle buffer (this may take a while): 9825 of 10000
I0000 00:00:1776180504.492090 47766246 shuffle_dataset_op.cc:483] Shuffle buffer filled.
W0000 00:00:1776180504.492226 47766246 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
I0000 00:00:1776180515.123087 47766836 shuffle_dataset_op.cc:453] ShuffleDatasetV3:268: Filling up shuffle buffer (this may take a while): 6173 of 10000
I0000 00:00:1776180520.393082 47766836 shuffle_dataset_op.cc:483] Shuffle buffer filled.
W0000 00:00:1776180520.393186 47766836 cache_dataset_ops.cc:912] The calling iterator did not fully read the dat

Communication round 1 of 1
Clients model training completed
Clients update aggregation completed


In [39]:
print("=== FedAvg + iDLG ===")
for i, metric_result in enumerate(fedavg_results):
    print(f"Metric result {i + 1}:")
    for sample_result in metric_result:
        print(f"  Input MSE:  {sample_result['input_mse']}")
        print(f"  Label MSE:  {sample_result['label_mse']}")

=== FedAvg + iDLG ===
Metric result 1:
  Input MSE:  7482.125
  Label MSE:  0.0


<h2> Comparison Matrix </h2>

In [40]:
all_results = {
    ("Per-FedAvg", "iDLG"): per_fedavg_results,
    ("FedAvg", "iDLG"): fedavg_results,
}

def compute_avg_metrics(results_list):
    total_input_mse = 0.0
    total_label_mse = 0.0
    count = 0
    for metric_result in results_list:
        for sample in metric_result:
            total_input_mse += sample["input_mse"]
            total_label_mse += sample["label_mse"]
            count += 1
    if count == 0:
        return float("nan"), float("nan")
    return total_input_mse / count, total_label_mse / count

algorithms = sorted(set(alg for alg, _ in all_results.keys()))
attacks = sorted(set(atk for _, atk in all_results.keys()))

header = f"{'Algorithm':<15}" + "".join(f"| {atk:^30}" for atk in attacks)
sub_header = f"{'':<15}" + "".join(f"| {'Input MSE':>13}  {'Label MSE':>13}" for _ in attacks)
separator = "-" * len(header)

print(header)
print(sub_header)
print(separator)

for alg in algorithms:
    row = f"{alg:<15}"
    for atk in attacks:
        key = (alg, atk)
        if key in all_results:
            avg_input, avg_label = compute_avg_metrics(all_results[key])
            row += f"| {avg_input:>13.4f}  {avg_label:>13.4f}"
        else:
            row += f"| {'N/A':>13}  {'N/A':>13}"
    print(row)

Algorithm      |              iDLG             
               |     Input MSE      Label MSE
-----------------------------------------------
FedAvg         |     7482.1250         0.0000
Per-FedAvg     |    11544.4092        16.0000
